# Chấm điểm động tác chống đẩy — Bước 8: mô hình cuối và chấm điểm một video

Bước 7 đã so sánh 5 mô hình deep learning. Mô hình được chọn là **1D-CNN trên đặc trưng hình học, train trên Mendeley + QEVD**:
trên Mendeley (chia theo người tập, 3 seed) nó đạt AUC 0.816, ngang gradient boosting (0.821) và là mô hình deep learning
duy nhất làm tốt ở cả góc trước (0.825) lẫn góc ngang (0.813).

Notebook này:
1. Train mô hình cuối trên **toàn bộ** dữ liệu (Mendeley nhãn sạch + QEVD), 3 seed, lấy trung bình dự đoán của 3 mô hình.
2. Với **mỗi video** được đưa vào, chạy cả pipeline: MediaPipe → nhận diện góc quay → đếm rep → cắt rep → mô hình → kết luận cho từng rep.
3. Xuất bảng kết quả và video có vẽ khung xương, số rep, kết luận cho từng video, kèm một bảng tổng hợp.

**Lưu ý về độ chính xác.** Mô hình cuối được train trên toàn bộ dữ liệu nên notebook này không đo được độ chính xác;
con số đáng tin là của bước 7. Nếu chấm thử một video của Mendeley thì đó là video mô hình đã thấy khi train,
chỉ để kiểm tra pipeline chạy đúng. Muốn kiểm tra thật phải dùng video mới.

Đầu vào: `pushup_dataset.npz` (bước 3), `qevd_dataset.npz` (bước 6) và một hoặc nhiều video `.mp4`.

## 1. Cấu hình

In [ ]:
VIDEOS = None            # None: chấm mọi file .mp4 trong /kaggle/input. Hoặc danh sách đường dẫn: ["/kaggle/input/.../a.mp4", ...]
VIEW = "auto"            # "auto": tự nhận diện góc quay cho từng video. Hoặc ép "front" / "side" / "diagonal" cho tất cả.
SEEDS = [0, 1, 2]        # số mô hình trong nhóm (ensemble)
EPOCHS = 40              # giống bước 7

# Kết luận cho mỗi rep dựa trên "điểm kết luận" p (xem mục 6 để biết điểm này lấy từ đầu ra nào của mô hình):
T_SAI = 0.60             # p >= T_SAI            -> "sai"
T_DUNG = 0.40            # p <= T_DUNG           -> "đúng"
                         # T_DUNG < p < T_SAI    -> "không chắc"
T_LOI = 0.50             # với rep "sai": loại lỗi nào có xác suất >= T_LOI thì được nêu tên

## 2. Đọc dữ liệu train

In [ ]:
import os, glob, random, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

def find_one(name):
    found = glob.glob(f"/kaggle/working/**/{name}", recursive=True) + glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert found, f"Không thấy {name}. Hãy Add Input dataset chứa file này."
    return found[0]

OUT_DIR = "/kaggle/working/cham_diem"
os.makedirs(OUT_DIR, exist_ok=True)

mend = np.load(find_one("pushup_dataset.npz"))
mX, m_good_bad, mview, msubject = mend["X2d"], mend["y"], mend["view"], mend["subject"]
VIEW_NAMES = ["front", "side", "diagonal"]
ERR_NAMES = ["sai", "nong", "hong", "sap"]

qevd = np.load(find_one("qevd_dataset.npz"))
quse = qevd["use"]
qX, qY = qevd["X2d"][quse], qevd["Y"][quse]
qview = np.zeros(len(qX), dtype=np.int64)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Mendeley:", mX.shape, "| QEVD:", qX.shape, "| thiết bị:", device)

## 3. Nhãn loại lỗi của Mendeley

Giống hệt bước 7: bỏ lỗi vị trí tay, chỉ giữ rep có nhãn chắc chắn.

In [ ]:
ERR = {
    1: ["nong"], 2: ["hong"], 3: ["hong"], 4: ["nong"], 5: ["nong", "hong"], 6: ["hong"], 7: ["sap"], 8: ["hong", "nong"],
    9: ["nong", "hong"], 10: ["hong"], 11: [], 12: ["sap"], 13: [], 14: [], 15: ["hong", "sap"], 16: ["nong"],
    17: ["sap"], 18: ["hong"], 19: ["hong", "sap"], 20: ["nong"], 21: ["nong"], 22: ["nong"], 23: ["hong"], 24: ["nong"],
}
NOT_ALL_BAD = [1, 3, 11, 12, 13, 15, 17]

mY = np.zeros((len(mX), 4), dtype=np.float32)
mY[:, 0] = m_good_bad
for i in np.where(m_good_bad == 1)[0]:
    for e in ERR[int(msubject[i])]:
        mY[i, ERR_NAMES.index(e)] = 1
only_hand_error = np.array([len(ERR[int(s)]) == 0 for s in msubject])
mclean = (m_good_bad == 0) | (~only_hand_error & ~np.isin(msubject, NOT_ALL_BAD))

print("Mendeley nhãn sạch:", int(mclean.sum()), "rep | đúng:", int((mY[mclean, 0] == 0).sum()), "| sai:", int(mY[mclean, 0].sum()),
      "| nong/hong/sap:", mY[mclean, 1:].sum(0).astype(int).tolist())
print("QEVD             :", len(qX), "rep | đúng:", int((qY[:, 0] == 0).sum()), "| sai:", int(qY[:, 0].sum()),
      "| nong/hong/sap:", qY[:, 1:].sum(0).astype(int).tolist())

## 4. Mô hình 1D-CNN đặc trưng

Giống hệt bước 7. Với mỗi khung hình, `frame_features` tính 16 đại lượng hình học (góc khuỷu, góc thân, góc gối, góc nách;
độ cao vai/hông/khuỷu; khoảng cách hai cổ tay, hai khuỷu, hai vai) cộng 12 kênh visibility; sau đó là 3 lớp tích chập theo thời gian.

In [ ]:
def torch_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = (ba * bc).sum(-1) / (ba.norm(dim=-1) * bc.norm(dim=-1) + 1e-6)
    return torch.acos(cos.clamp(-1 + 1e-6, 1 - 1e-6)) / np.pi           # về khoảng 0..1


def frame_features(x):
    # x: (B, T, 12, 3) -> (B, 28, T): 16 đặc trưng hình học + 12 kênh visibility cho từng khung
    P, V = x[..., :2], x[..., 2]
    f = []
    for s in (0, 1):                                                    # trái, phải
        sh, el, wr, hip, kn, an = [P[:, :, j + s] for j in (0, 2, 4, 6, 8, 10)]
        f += [torch_angle(sh, el, wr), torch_angle(sh, hip, an), torch_angle(hip, kn, an), torch_angle(el, sh, hip)]
    y = lambda joints: P[:, :, joints, 1].mean(2)
    sh_y, el_y, wr_y, hip_y = y([0, 1]), y([2, 3]), y([4, 5]), y([6, 7])
    f += [wr_y - sh_y, hip_y - sh_y, el_y - sh_y,
          sh_y - sh_y.mean(1, keepdim=True), hip_y - hip_y.mean(1, keepdim=True),
          (P[:, :, 4] - P[:, :, 5]).norm(dim=-1), (P[:, :, 2] - P[:, :, 3]).norm(dim=-1), (P[:, :, 0] - P[:, :, 1]).norm(dim=-1)]
    return torch.cat([torch.stack(f, dim=1), V.permute(0, 2, 1)], dim=1)


class FeatCNN(nn.Module):
    def __init__(self, n_out=4, c=64, dropout=0.3):
        super().__init__()
        self.input_bn = nn.BatchNorm1d(28)
        self.net = nn.Sequential(
            nn.Conv1d(28, c, 5, padding=2), nn.BatchNorm1d(c), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(c, c, 5, padding=2), nn.BatchNorm1d(c), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(c, c, 3, padding=1), nn.BatchNorm1d(c), nn.ReLU())
        self.head = nn.Sequential(nn.Dropout(dropout), nn.Linear(2 * c + 3, n_out))

    def forward(self, x, view_id):
        h = self.net(self.input_bn(frame_features(x)))
        return self.head(torch.cat([h.mean(2), h.amax(2), nn.functional.one_hot(view_id, 3).float()], dim=1))

## 5. Train mô hình cuối

Train trên toàn bộ Mendeley nhãn sạch + QEVD, mỗi seed một mô hình, không tăng cường dữ liệu, số epoch cố định (như bước 7).
Khi dự đoán, lấy trung bình xác suất của các mô hình.

In [ ]:
train_X = np.concatenate([mX[mclean], qX])
train_V = np.concatenate([mview[mclean], qview])
train_Y = np.concatenate([mY[mclean], qY])
print("Số rep train:", len(train_X), "| tỉ lệ sai:", round(float(train_Y[:, 0].mean()), 3))


def train_final(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    model = FeatCNN().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
    data = torch.utils.data.TensorDataset(torch.from_numpy(train_X), torch.from_numpy(train_V), torch.from_numpy(train_Y))
    loader = torch.utils.data.DataLoader(data, batch_size=64, shuffle=True)
    for epoch in range(EPOCHS):
        model.train()
        for xb, vb, yb in loader:
            opt.zero_grad()
            nn.functional.binary_cross_entropy_with_logits(model(xb.to(device), vb.to(device)), yb.to(device)).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        scheduler.step()
    return model.eval()


models = []
for seed in SEEDS:
    t0 = time.time()
    model = train_final(seed)
    torch.save(model.state_dict(), os.path.join(OUT_DIR, f"featcnn_seed{seed}.pt"))
    models.append(model)
    print(f"seed {seed}: {time.time() - t0:.0f}s")


@torch.no_grad()
def predict_reps(X, view_id):
    x = torch.from_numpy(X).to(device)
    v = torch.full((len(X),), view_id, dtype=torch.long, device=device)
    return torch.stack([torch.sigmoid(m(x, v)) for m in models]).mean(0).cpu().numpy()     # (số rep, 4)

## 6. Quy tắc kết luận và ý nghĩa của ngưỡng

Mô hình cho 4 xác suất cho mỗi rep: `sai` (sai nói chung), `nong`, `hong`, `sap`. **Điểm kết luận** được lấy như sau:

- **Góc ngang, góc chéo:** điểm = xác suất `sai`.
- **Góc trước:** điểm = giá trị lớn hơn giữa `nong` và `sap`.

Lý do của quy tắc riêng cho góc trước: ở góc này hông bị thân che, nên đầu ra `hong` chỉ là đoán (AUC khoảng 0,6 ở bước 7)
và nó kéo theo cả đầu ra `sai`. Thử nghiệm với webcam thật cho thấy một người tập đúng 10 rep bị báo sai 9 rep,
trong khi `nong` và `sap` đều rất thấp, chỉ có `hong` cao. Vì vậy ở góc trước hệ thống chỉ xét hai lỗi nhìn thấy được;
lỗi về hông không được đánh giá ở góc này.

Nếu notebook bước 7 được gắn làm Input (file `du_doan.npz`), cell dưới dùng dự đoán của 1D-CNN trên các fold test để:
1. So sánh hai cách lấy điểm ở góc trước (trên QEVD và trên Mendeley góc trước).
2. Cho thấy ngưỡng ảnh hưởng thế nào tới tỉ lệ bắt được rep sai và tỉ lệ báo nhầm rep đúng.

Đề tài ưu tiên ít báo nhầm, nên mặc định chỉ kết luận "sai" khi điểm từ 0.60 trở lên.
Không có file đó thì cell này bỏ qua, không ảnh hưởng các bước sau.

In [ ]:
def verdict_score(p, view):
    # p: (..., 4) = [sai, nong, hong, sap]. Góc trước chỉ xét hai lỗi nhìn thấy được.
    return np.maximum(p[..., 1], p[..., 3]) if view == "front" else p[..., 0]


found = glob.glob("/kaggle/input/**/du_doan.npz", recursive=True) + glob.glob("/kaggle/working/**/du_doan.npz", recursive=True)
if found:
    d7 = np.load(found[0])
    keys_A = [k for k in d7.files if k.startswith("A_featcnn")]
    keys_B = [k for k in d7.files if k.startswith("B_featcnn_qevd")]
    PA, PB = np.mean([d7[k] for k in keys_A], axis=0), np.mean([d7[k] for k in keys_B], axis=0)
    qY7, mY7, clean7, view7 = d7["qY"], d7["mY"], d7["mclean"], d7["mview"]

    def compare_front_rule(name, P, Y):
        good = Y[:, 0] == 0
        seen = (Y[:, 1] == 1) | (Y[:, 3] == 1)                 # rep có lỗi nhìn thấy được ở góc trước
        hip_only = (Y[:, 2] == 1) & ~seen
        rows = {}
        for label, score in [("điểm = sai (cách cũ)", P[:, 0]), ("điểm = max(nong, sap)", verdict_score(P, "front"))]:
            q = score >= T_SAI
            rows[label] = {"báo nhầm rep đúng": q[good].mean(), "bắt được rep nong/sap": q[seen].mean(),
                           "bắt được rep chỉ lỗi hông": q[hip_only].mean() if hip_only.any() else np.nan}
        print(f"{name}: {len(Y)} rep ({int(good.sum())} đúng, {int(seen.sum())} có nong/sap, {int(hip_only.sum())} chỉ lỗi hông), ngưỡng {T_SAI}")
        display(pd.DataFrame(rows).T.round(3))

    print("1) Hai cách lấy điểm ở góc trước")
    compare_front_rule("QEVD (toàn góc trước)", PA, qY7)
    front = clean7 & (view7 == 0)
    compare_front_rule("Mendeley góc trước", PB[front], mY7[front])

    # Điểm kết luận đúng như lúc chấm video: góc trước dùng max(nong, sap), góc khác dùng sai
    score = np.where(view7 == 0, verdict_score(PB, "front"), verdict_score(PB, "side"))[clean7]
    y7 = mY7[clean7, 0]
    rows = []
    for t in [0.30, 0.40, 0.50, 0.60, 0.70]:
        q = score >= t
        rows.append({"ngưỡng": t, "accuracy": (q == (y7 == 1)).mean(), "recall": q[y7 == 1].mean(), "báo nhầm": q[y7 == 0].mean()})
    print(f"2) Ảnh hưởng của ngưỡng, trên {len(y7)} rep Mendeley nhãn sạch (cả 3 góc, trung bình {len(keys_B)} seed):")
    display(pd.DataFrame(rows).set_index("ngưỡng").round(3))

    sai, dung = score >= T_SAI, score <= T_DUNG
    print(f"Với T_SAI = {T_SAI}, T_DUNG = {T_DUNG}:")
    print(f"  kết luận 'sai'       : {sai.mean():.1%} số rep, trong đó {y7[sai].mean():.1%} thật sự sai")
    print(f"  kết luận 'đúng'      : {dung.mean():.1%} số rep, trong đó {(1 - y7[dung]).mean():.1%} thật sự đúng")
    print(f"  kết luận 'không chắc': {(~sai & ~dung).mean():.1%} số rep")
else:
    print("Không thấy du_doan.npz (output của bước 7): bỏ qua phần phân tích.")

## 7. Cài MediaPipe và tải mô hình pose

Cùng mô hình `pose_landmarker_full` đã dùng để tạo dữ liệu train (bắt buộc phải cùng mô hình).

In [ ]:
!pip install -q mediapipe

MODEL_PATH = "/kaggle/working/pose_landmarker_full.task"
if not os.path.exists(MODEL_PATH):
    !wget -q -O "{MODEL_PATH}" "https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_full/float16/latest/pose_landmarker_full.task"

import cv2
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision
print("MediaPipe:", mp.__version__)

## 8. Các hàm của pipeline

Trích keypoint (như bước 1), đếm rep (bước 2), chuẩn hoá và đưa rep về 32 khung (bước 3).

In [ ]:
def extract_video(video_path):
    options = vision.PoseLandmarkerOptions(
        base_options=python.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=vision.RunningMode.VIDEO, num_poses=1)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 15.0
    width, height = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    kp2d, kp3d, detected = [], [], []
    with vision.PoseLandmarker.create_from_options(options) as landmarker:
        i = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            image = mp.Image(image_format=mp.ImageFormat.SRGB, data=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            result = landmarker.detect_for_video(image, int(i * 1000 / fps))
            i += 1
            if result.pose_landmarks:
                kp2d.append([[l.x, l.y, l.z, l.visibility] for l in result.pose_landmarks[0]])
                kp3d.append([[l.x, l.y, l.z, l.visibility] for l in result.pose_world_landmarks[0]])
                detected.append(True)
            else:
                kp2d.append(np.zeros((33, 4))); kp3d.append(np.zeros((33, 4))); detected.append(False)
    cap.release()
    return dict(kp2d=np.asarray(kp2d, dtype=np.float32).reshape(-1, 33, 4),
                kp3d=np.asarray(kp3d, dtype=np.float32).reshape(-1, 33, 4),
                detected=np.asarray(detected, dtype=bool), fps=fps, width=width, height=height)

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import find_peaks, savgol_filter

VIS_THR = 0.5
MIN_AMP = {"front": 0.10, "side": 3.0, "diagonal": 3.0}     # biên độ tối thiểu của một rep, theo đơn vị của tín hiệu
L_SH, R_SH, L_EL, R_EL, L_WR, R_WR = 11, 12, 13, 14, 15, 16
L_HIP, L_ANK = 23, 27


def joint_angle(a, b, c):
    ba, bc = a - b, c - b
    cos = (ba * bc).sum(-1) / (np.linalg.norm(ba, axis=-1) * np.linalg.norm(bc, axis=-1) + 1e-8)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def fill_gaps(x, valid):
    # Nội suy tuyến tính các khung không hợp lệ
    x = x.astype(np.float64).copy()
    idx = np.arange(len(x))
    if valid.sum() >= 2:
        x[~valid] = np.interp(idx[~valid], idx[valid], x[valid])
    return x


def near_side(vis):
    # 0 = bên trái, 1 = bên phải: bên nào tay nhìn rõ hơn
    return 0 if vis[:, [L_SH, L_EL, L_WR]].mean() >= vis[:, [R_SH, R_EL, R_WR]].mean() else 1


def rep_signal(d, view):
    kp = d["kp2d"]
    px = kp[:, :, :2] * np.array([int(d["width"]), int(d["height"])])   # về pixel để góc không bị méo
    vis, det = kp[:, :, 3], d["detected"]

    def in_frame(j):
        return (kp[:, j, 0] > -0.1) & (kp[:, j, 0] < 1.1) & (kp[:, j, 1] > -0.1) & (kp[:, j, 1] < 1.1)

    if view == "front":
        sh_y = px[:, [L_SH, R_SH], 1].mean(1)
        wr_y = px[:, [L_WR, R_WR], 1].mean(1)
        sh_w = np.linalg.norm(px[:, L_SH] - px[:, R_SH], axis=-1)
        valid = det & (vis[:, [L_SH, R_SH]].min(1) > VIS_THR) & (vis[:, [L_WR, R_WR]].max(1) > VIS_THR) \
            & in_frame(L_SH) & in_frame(R_SH)
        scale = np.median(sh_w[valid]) if valid.any() else 1.0
        sig = (wr_y - sh_y) / (scale + 1e-6)
    else:
        side = near_side(vis)
        s, e, w = L_SH + side, L_EL + side, L_WR + side
        sig = joint_angle(px[:, s], px[:, e], px[:, w])
        valid = det & (vis[:, [s, e, w]].min(1) > VIS_THR) & in_frame(s) & in_frame(e) & in_frame(w)

    sig = fill_gaps(sig, valid)
    if len(sig) >= 7:
        sig = savgol_filter(sig, 7, 2)      # làm mượt nhẹ
    return sig, valid


def find_reps(sig, valid, fps, min_amp=0.0, min_prom=0.35, min_rep_s=0.8, max_rep_s=8.0, min_valid=0.7):
    if valid.mean() < 0.3:      # video gần như không nhìn rõ tay: bỏ cả video
        return []
    lo, hi = np.percentile(sig[valid], [5, 95])
    rng = hi - lo
    bottoms, _ = find_peaks(-sig, prominence=min_prom * rng, distance=max(1, int(min_rep_s * fps * 0.6)))
    half = int(max_rep_s * fps / 2)
    reps = []
    for i, b in enumerate(bottoms):
        left = bottoms[i - 1] if i > 0 else max(0, b - half)
        right = bottoms[i + 1] if i + 1 < len(bottoms) else min(len(sig) - 1, b + half)
        start = left + int(np.argmax(sig[left:b + 1]))
        end = b + int(np.argmax(sig[b:right + 1]))
        dur = (end - start) / fps
        amp = float(min(sig[start], sig[end]) - sig[b])       # tín hiệu hạ xuống bao nhiêu so với đỉnh thấp hơn
        deep = amp > 0.25 * rng and amp >= min_amp
        valid_frac = float(valid[start:end + 1].mean()) if end > start else 0.0
        reps.append(dict(start=int(start), bottom=int(b), end=int(end), dur=round(dur, 2),
                         valid_frac=round(valid_frac, 2), amp=round(amp, 3),
                         ok=bool(min_rep_s <= dur <= max_rep_s and deep and valid_frac >= min_valid)))
    return reps

In [ ]:
T = 32
JOINTS = [11, 12, 13, 14, 15, 16, 23, 24, 25, 26, 27, 28]
JOINT_NAMES = ["L_shoulder", "R_shoulder", "L_elbow", "R_elbow", "L_wrist", "R_wrist",
               "L_hip", "R_hip", "L_knee", "R_knee", "L_ankle", "R_ankle"]
# Cạnh của đồ thị khung xương (chỉ số theo JOINTS), dùng cho ST-GCN và để vẽ
EDGES = [(0, 1), (0, 2), (2, 4), (1, 3), (3, 5), (0, 6), (1, 7), (6, 7), (6, 8), (8, 10), (7, 9), (9, 11)]
SH, HIP = [0, 1], [6, 7]
VIEW_ID = {"front": 0, "side": 1, "diagonal": 2}
LABEL_ID = {"good": 0, "bad": 1}

In [ ]:
def resample(seq, n):
    pos = np.linspace(0, len(seq) - 1, n)
    lo = np.floor(pos).astype(int)
    hi = np.minimum(lo + 1, len(seq) - 1)
    w = (pos - lo).reshape(-1, *[1] * (seq.ndim - 1))
    return seq[lo] * (1 - w) + seq[hi] * w


def normalize_2d(xy):
    mid_sh, mid_hip = xy[:, SH].mean(1), xy[:, HIP].mean(1)
    origin = np.median(mid_hip, axis=0)
    torso = np.median(np.linalg.norm(mid_sh - mid_hip, axis=-1))
    sh_width = np.median(np.linalg.norm(xy[:, SH[0]] - xy[:, SH[1]], axis=-1))
    out = (xy - origin) / (max(torso, sh_width) + 1e-6)
    return np.clip(out, -3, 3)      # chặn các khớp bị MediaPipe đoán văng ra xa (99% giá trị nằm trong ±1.8)


def make_sample(xy, xyz, vis, start, end):
    sl = slice(start, end + 1)
    v = resample(vis[sl], T)[..., None]
    x2d = np.concatenate([resample(normalize_2d(xy[sl]), T), v], axis=-1)     # (T, 12, 3): x, y, visibility
    x3d = np.concatenate([resample(xyz[sl], T), v], axis=-1)                  # (T, 12, 4): x, y, z, visibility
    return x2d.astype(np.float32), x3d.astype(np.float32)

## 9. Nhận diện góc quay

Luật hình học: tỉ lệ giữa **độ rộng vai** và **chiều dài thân** (vai tới hông) trên ảnh, lấy trung vị cả video.
- Góc ngang: hai vai gần chồng nhau nên tỉ lệ rất nhỏ.
- Góc trước: vai mở rộng, thân bị co ngắn do phối cảnh nên tỉ lệ lớn.

Ngưỡng được chọn từ 144 video Mendeley (tỉ lệ trung vị: ngang 0.17, chéo 0.54, trước 2.46) và clip QEVD (trước 1.62):
dưới 0.28 là góc ngang, trên 0.95 là góc trước, còn lại là góc chéo. Trên Mendeley luật này đúng khoảng 95% số video.

In [ ]:
def detect_view(d):
    kp, det = d["kp2d"], d["detected"]
    px = kp[:, :, :2] * np.array([d["width"], d["height"]])
    ok = det & (kp[:, [11, 12, 23, 24], 3].min(1) > 0.5)
    if ok.sum() < 5:
        return "front", float("nan")
    shoulder_w = np.linalg.norm(px[ok, 11] - px[ok, 12], axis=-1)
    torso = np.linalg.norm(px[ok][:, [11, 12]].mean(1) - px[ok][:, [23, 24]].mean(1), axis=-1)
    ratio = float(np.median(shoulder_w / (torso + 1e-6)))
    return ("side" if ratio < 0.28 else "front" if ratio > 0.95 else "diagonal"), ratio

## 10. Hàm chấm điểm một video

Dao động có biên độ quá nhỏ hoặc quá ngắn/dài không được tính là rep (người đứng yên ở tư thế chờ sẽ ra 0 rep).

Với mỗi rep tìm được:
- Rep mà dưới 70% khung hình nhìn rõ tay, hoặc quá nông so với các rep khác trong video → **không đánh giá được**.
- Còn lại, mô hình cho 4 xác suất; **điểm kết luận** được lấy theo quy tắc ở mục 6 (góc trước: lớn hơn giữa `nong` và `sap`;
  góc khác: `sai`) → **sai / đúng / không chắc** theo các ngưỡng ở mục 1.
- Với rep "sai", nêu các loại lỗi có xác suất từ `T_LOI` trở lên. Ở góc trước, lỗi về hông (`hong`) không được nêu và
  không được tính vào điểm, vì hông bị thân che.

Hàm `render_video` vẽ khung xương (xanh: nhìn rõ, đỏ: visibility thấp), số rep đã hoàn thành và kết luận của rep vừa xong.
Chữ trên video viết không dấu vì OpenCV không vẽ được tiếng Việt có dấu.

In [ ]:
ERR_TEXT = {"nong": "xuống không đủ sâu", "hong": "hông võng / mông nhô", "sap": "đổ sập / chân chạm sàn"}
COLUMNS = ["rep", "bắt_đầu_s", "kết_thúc_s", "start", "end", "kết_luận", "lỗi", "điểm", "p_sai", "p_nong", "p_hong", "p_sap"]


def score_video(video_path):
    d = extract_video(video_path)
    view, ratio = (VIEW, float("nan")) if VIEW != "auto" else detect_view(d)

    sig, valid = rep_signal(d, view)
    candidates = find_reps(sig, valid, d["fps"], MIN_AMP[view])
    # Dao động quá nhỏ hoặc quá ngắn/dài không phải là một rep (ví dụ người đang đứng yên ở tư thế chờ): bỏ hẳn, không đếm.
    reps = [r for r in candidates if r["amp"] >= MIN_AMP[view] and 0.8 <= r["dur"] <= 8.0]

    # Keypoint 12 khớp theo pixel, nội suy khung không phát hiện được (như bước 3)
    det = d["detected"]
    xy = d["kp2d"][:, JOINTS, :2] * np.array([d["width"], d["height"]], dtype=np.float32)
    xyz = d["kp3d"][:, JOINTS, :3].copy()
    vis = d["kp2d"][:, JOINTS, 3].copy()
    if det.sum() >= 2:
        idx = np.arange(len(det))
        for arr in (xy, xyz):
            flat = arr.reshape(len(det), -1)
            for k in range(flat.shape[1]):
                flat[~det, k] = np.interp(idx[~det], idx[det], flat[det, k])
    vis[~det] = 0.0

    ok_reps = [r for r in reps if r["ok"]]
    if ok_reps:
        X = np.stack([make_sample(xy, xyz, vis, r["start"], r["end"])[0] for r in ok_reps])
        prob = predict_reps(X, VIEW_NAMES.index(view))

    rows, k = [], 0
    for i, r in enumerate(reps, 1):
        row = dict(rep=i, bắt_đầu_s=round(r["start"] / d["fps"], 1), kết_thúc_s=round(r["end"] / d["fps"], 1), start=r["start"], end=r["end"])
        if not r["ok"]:
            row.update(kết_luận="không đánh giá được", lỗi="", điểm=np.nan, p_sai=np.nan, p_nong=np.nan, p_hong=np.nan, p_sap=np.nan)
        else:
            p = prob[k]; k += 1
            score = float(verdict_score(p, view))
            verdict = "sai" if score >= T_SAI else "đúng" if score <= T_DUNG else "không chắc"
            errors = []
            if verdict == "sai":
                for j, e in enumerate(ERR_NAMES[1:], 1):
                    if p[j] >= T_LOI and not (e == "hong" and view == "front"):
                        errors.append(ERR_TEXT[e])
            row.update(kết_luận=verdict, lỗi=", ".join(errors), điểm=score, p_sai=p[0], p_nong=p[1], p_hong=p[2], p_sap=p[3])
        rows.append(row)
    return d, view, ratio, pd.DataFrame(rows, columns=COLUMNS)     # có sẵn tên cột kể cả khi video không có rep nào


SKELETON = [(JOINTS[a], JOINTS[b]) for a, b in EDGES]
LABEL = {"đúng": ("DUNG", (0, 200, 0)), "sai": ("SAI", (0, 0, 255)), "không chắc": ("KHONG CHAC", (0, 200, 255)),
         "không đánh giá được": ("KHONG DANH GIA DUOC", (160, 160, 160))}
ERR_ASCII = {"xuống không đủ sâu": "xuong chua du sau", "hông võng / mông nhô": "hong vong / mong nho", "đổ sập / chân chạm sàn": "do sap / chan cham san"}


def render_video(video_path, d, view, result, out_path):
    end_frame = {int(r.end): r for r in result.itertuples()}
    cap = cv2.VideoCapture(video_path)
    scale = 720 / max(d["width"], d["height"])
    w, h = int(d["width"] * scale) // 2 * 2, int(d["height"] * scale) // 2 * 2
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*"mp4v"), d["fps"], (w, h))
    count, last, snapshots = 0, None, []
    for i in range(len(d["detected"])):
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.resize(frame, (w, h))
        if d["detected"][i]:
            p, v = d["kp2d"][i, :, :2] * np.array([w, h]), d["kp2d"][i, :, 3]
            for a, b in SKELETON:
                color = (0, 220, 0) if min(v[a], v[b]) > 0.5 else (0, 0, 255)
                cv2.line(frame, tuple(p[a].astype(int)), tuple(p[b].astype(int)), color, 2)
        if i in end_frame:
            count, last = count + 1, end_frame[i]
        cv2.rectangle(frame, (0, 0), (w, 78), (0, 0, 0), -1)
        cv2.putText(frame, f"Rep: {count}   goc quay: {view}", (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
        if last is not None:
            text, color = LABEL[last.kết_luận]
            errors = ", ".join(ERR_ASCII[e] for e in last.lỗi.split(", ") if e)
            cv2.putText(frame, f"Rep {last.rep}: {text}" + (f" ({errors})" if errors else ""), (10, 62), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
            if i == min(int(last.end) + 2, len(d["detected"]) - 1):
                snapshots.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        writer.write(frame)
    cap.release(); writer.release()
    return snapshots

## 11. Chấm tất cả video

Mỗi video cho ra: một dòng thông tin (số khung, góc quay nhận diện được), bảng kết luận từng rep, vài khung hình mẫu
(chụp ngay sau khi mỗi rep kết thúc), và hai file trong thư mục `cham_diem`: `<tên video>_cham_diem.mp4`, `<tên video>_ket_qua.csv`.
Cuối cùng là bảng tổng hợp mọi video.

In [ ]:
import matplotlib.pyplot as plt

if VIDEOS is None:
    VIDEOS = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True))
    assert VIDEOS, "Không thấy file .mp4 nào trong /kaggle/input. Hãy Add Input một dataset có video, hoặc đặt VIDEOS."
print("Số video:", len(VIDEOS))

summary, all_results = [], []
for video_path in VIDEOS:
    name = os.path.splitext(os.path.basename(video_path))[0]
    print("\n" + "=" * 100)
    print("Video:", video_path)
    d, view, ratio, result = score_video(video_path)
    print(f"{len(d['detected'])} khung, {d['fps']:.0f} fps, {d['width']}x{d['height']} | phát hiện người: {d['detected'].mean():.0%}"
          f" | góc quay: {view} (tỉ lệ vai/thân = {ratio:.2f}) | số rep: {len(result)}")

    shown = result.drop(columns=["start", "end"])
    shown.to_csv(os.path.join(OUT_DIR, f"{name}_ket_qua.csv"), index=False)
    all_results.append(shown.assign(video=name))
    if len(result):
        display(shown.round(2))
    if view == "front":
        print("Góc trước: lỗi về hông không được đánh giá. Quay từ bên hông nếu muốn kiểm tra lỗi này.")

    snapshots = render_video(video_path, d, view, result, os.path.join(OUT_DIR, f"{name}_cham_diem.mp4"))
    if snapshots:
        show = snapshots[:6]
        fig, axes = plt.subplots(1, len(show), figsize=(4.5 * len(show), 5), squeeze=False)
        for ax, img in zip(axes[0], show):
            ax.imshow(img); ax.axis("off")
        plt.tight_layout(); plt.show()

    counts = result["kết_luận"].value_counts()
    summary.append({"video": name, "góc quay": view, "phát hiện người": round(float(d["detected"].mean()), 2), "số rep": len(result),
                    **{k: int(counts.get(k, 0)) for k in ["đúng", "sai", "không chắc", "không đánh giá được"]}})

print("\n" + "=" * 100)
print("Tổng hợp:")
summary = pd.DataFrame(summary).set_index("video")
summary.to_csv(os.path.join(OUT_DIR, "tong_hop.csv"))
pd.concat(all_results).to_csv(os.path.join(OUT_DIR, "ket_qua_rep.csv"), index=False)
display(summary)

## 12. Kết quả

Trong thư mục `cham_diem`:
- `<tên video>_cham_diem.mp4`: video có chú thích (tải về để xem; trình duyệt có thể không phát trực tiếp định dạng này).
- `<tên video>_ket_qua.csv`: bảng kết luận từng rep của video đó.
- `ket_qua_rep.csv`, `tong_hop.csv`: kết quả gộp và bảng tổng hợp của mọi video.
- `featcnn_seed*.pt`: trọng số mô hình cuối.

In [ ]:
!ls -lh "{OUT_DIR}"